# Sensor Lsoa Context Join

Converted from the original Python workflow script so the dissertation repository uses notebook-based workflow artefacts.


In [ ]:
#!/usr/bin/env python3
"""Build LSOA 2021 context joins for dissertation Vivacity countlines.

The synchronized spatial unit is LSOA 2021. Vivacity countline geometries
are point-joined to LSOA 2021 polygons, Census 2021 tables are already at
LSOA 2021, and IMD 2019 is bridged from LSOA 2011 to LSOA 2021 using the
exact-fit lookup prepared in context_data/processed/imd.
"""

from __future__ import annotations

import csv
import json
import math
import re
from collections import defaultdict
from pathlib import Path


BASE = Path("/Users/lu_nanxi/CASA/Dissertation_Data")
CONTEXT = BASE / "context_data"
RAW_META = CONTEXT / "raw" / "vivacity_metadata"
PROCESSED = CONTEXT / "processed"
SENSOR_OUT = PROCESSED / "sensors"
QA_OUT = CONTEXT / "quality_checks"

SENSOR_QA = BASE / "Vivacity_R_outputs" / "tables" / "sensor_qa_table.csv"
DAILY_SENSOR_MAIN = BASE / "Vivacity_R_outputs" / "tables" / "sensor_daily_main_analysis.csv"
COUNTLINES_JSON = RAW_META / "countlines.json"
HARDWARE_TXT = RAW_META / "hardware_positions_with_status.txt"
LSOA_GEOJSON = PROCESSED / "lsoa" / "lcr_lsoa_2021_boundaries.geojson"
LSOA_LOOKUP = PROCESSED / "lsoa" / "lcr_lsoa_2021_lookup.csv"
IMD_LSOA21 = PROCESSED / "imd" / "imd_2019_lcr_joined_to_lsoa2021_exact_fit.csv"
CENSUS_DIR = PROCESSED / "census_lsoa"


def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def write_csv(path: Path, rows: list[dict[str, object]], fieldnames: list[str] | None = None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        keys: list[str] = []
        for row in rows:
            for key in row:
                if key not in keys:
                    keys.append(key)
        fieldnames = keys
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def to_float(value: object) -> float | None:
    if value in (None, "", "NULL"):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def to_int(value: object) -> int | None:
    number = to_float(value)
    if number is None:
        return None
    return int(round(number))


def slug(value: str) -> str:
    value = value.lower()
    value = value.replace("%", "pct")
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def safe_div(numerator: float | int | None, denominator: float | int | None) -> float | None:
    if numerator is None or denominator in (None, 0):
        return None
    return 100.0 * float(numerator) / float(denominator)


def parse_hardware() -> list[dict]:
    if not HARDWARE_TXT.exists():
        return []
    text = HARDWARE_TXT.read_text(encoding="utf-8")
    # The file was captured as "200\n{json}" to preserve the API status code.
    json_start = text.find("{")
    if json_start == -1:
        return []
    payload = json.loads(text[json_start:])
    return payload.get("data", [])


def point_in_ring(lon: float, lat: float, ring: list[list[float]]) -> bool:
    inside = False
    n = len(ring)
    if n < 3:
        return False
    j = n - 1
    for i in range(n):
        xi, yi = ring[i][0], ring[i][1]
        xj, yj = ring[j][0], ring[j][1]
        intersects = ((yi > lat) != (yj > lat)) and (
            lon < (xj - xi) * (lat - yi) / ((yj - yi) or 1e-15) + xi
        )
        if intersects:
            inside = not inside
        j = i
    return inside


def point_in_polygon(lon: float, lat: float, coordinates: list) -> bool:
    if not coordinates:
        return False
    outer = coordinates[0]
    if not point_in_ring(lon, lat, outer):
        return False
    for hole in coordinates[1:]:
        if point_in_ring(lon, lat, hole):
            return False
    return True


def point_in_geometry(lon: float, lat: float, geometry: dict) -> bool:
    geom_type = geometry.get("type")
    coords = geometry.get("coordinates", [])
    if geom_type == "Polygon":
        return point_in_polygon(lon, lat, coords)
    if geom_type == "MultiPolygon":
        return any(point_in_polygon(lon, lat, polygon) for polygon in coords)
    return False


def line_midpoint(coords: list[list[float]]) -> tuple[float | None, float | None]:
    if not coords:
        return None, None
    lon = sum(point[0] for point in coords) / len(coords)
    lat = sum(point[1] for point in coords) / len(coords)
    return lon, lat


def build_countline_locations() -> list[dict[str, object]]:
    qa_rows = read_csv(SENSOR_QA)
    target_ids = {int(row["countline_id"]) for row in qa_rows}
    qa_by_id = {int(row["countline_id"]): row for row in qa_rows}

    countlines = json.loads(COUNTLINES_JSON.read_text(encoding="utf-8")).get("data", [])
    countline_by_id = {int(row["id"]): row for row in countlines if int(row.get("id", -1)) in target_ids}

    hardware_by_viewpoint: dict[int, dict] = {}
    for hardware in parse_hardware():
        child_entities = hardware.get("child_entities") or {}
        for viewpoint_id in child_entities.get("viewpoint_ids") or []:
            hardware_by_viewpoint[int(viewpoint_id)] = hardware

    rows: list[dict[str, object]] = []
    for countline_id in sorted(target_ids):
        qa = qa_by_id[countline_id]
        countline = countline_by_id.get(countline_id, {})
        gps = (countline.get("geometry") or {}).get("gps") or {}
        midpoint_lon, midpoint_lat = line_midpoint(gps.get("coordinates") or [])
        viewpoint_id = countline.get("viewpoint_id")
        hardware = hardware_by_viewpoint.get(int(viewpoint_id)) if viewpoint_id is not None else None
        hardware_location = (hardware or {}).get("location") or {}
        local_bng = ((hardware or {}).get("local_coordinates") or {}).get("BNG") or {}
        sensor_match = re.search(r"\b[sS](\d+)", qa.get("countline_name", ""))

        rows.append(
            {
                "scheme_id": qa.get("scheme_id"),
                "road_group": qa.get("road_group"),
                "countline_id": countline_id,
                "countline_name": qa.get("countline_name"),
                "route_type": qa.get("route_type"),
                "installation_month": qa.get("installation_month"),
                "first_reliable_date": qa.get("first_reliable_date"),
                "raw_first_date": qa.get("raw_first_date"),
                "raw_last_date": qa.get("raw_last_date"),
                "vivacity_viewpoint_id": viewpoint_id,
                "parsed_sensor_number": int(sensor_match.group(1)) if sensor_match else "",
                "countline_midpoint_lon": midpoint_lon,
                "countline_midpoint_lat": midpoint_lat,
                "hardware_id": (hardware or {}).get("id", ""),
                "hardware_sensor_number": (hardware or {}).get("sensor_number", ""),
                "hardware_name": (hardware or {}).get("name", ""),
                "hardware_status": (hardware or {}).get("status", ""),
                "hardware_lon": hardware_location.get("longitude", ""),
                "hardware_lat": hardware_location.get("latitude", ""),
                "hardware_bng_easting": local_bng.get("easting", ""),
                "hardware_bng_northing": local_bng.get("northing", ""),
                "coordinate_source": "Vivacity countline GPS midpoint",
            }
        )
    return rows


def spatial_join_lsoa(sensor_rows: list[dict[str, object]]) -> list[dict[str, object]]:
    geo = json.loads(LSOA_GEOJSON.read_text(encoding="utf-8"))
    joined: list[dict[str, object]] = []
    for row in sensor_rows:
        lon = to_float(row.get("countline_midpoint_lon"))
        lat = to_float(row.get("countline_midpoint_lat"))
        match = None
        if lon is not None and lat is not None:
            for feature in geo["features"]:
                if point_in_geometry(lon, lat, feature["geometry"]):
                    match = feature["properties"]
                    break
        out = dict(row)
        out.update(
            {
                "analysis_spatial_unit": "LSOA 2021",
                "analysis_lsoa21cd": (match or {}).get("LSOA21CD", ""),
                "analysis_lsoa21nm": (match or {}).get("LSOA21NM", ""),
                "lsoa21_rural_urban_class": (match or {}).get("RUC21NM", ""),
                "lsoa21_urban_rural": (match or {}).get("Urban_rura", ""),
                "spatial_join_success": bool(match),
            }
        )
        joined.append(out)
    return joined


def build_lsoa_lookup() -> dict[str, dict[str, str]]:
    lookup = {}
    geo = json.loads(LSOA_GEOJSON.read_text(encoding="utf-8"))
    for feature in geo["features"]:
        props = feature.get("properties") or {}
        code = props.get("LSOA21CD")
        if code:
            lookup[code] = props
    return lookup


def build_imd_lsoa21() -> tuple[list[dict[str, object]], dict[str, dict[str, object]]]:
    rows = read_csv(IMD_LSOA21)
    grouped: dict[str, list[dict[str, str]]] = defaultdict(list)
    for row in rows:
        grouped[row["LSOA21CD"]].append(row)

    weight_col = "Total population: mid 2015 (excluding prisoners)"
    wanted = {
        "Index of Multiple Deprivation (IMD) Score": "imd_score",
        "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)": "imd_rank",
        "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)": "imd_decile",
        "Income Score (rate)": "income_score",
        "Income Decile (where 1 is most deprived 10% of LSOAs)": "income_decile",
        "Employment Score (rate)": "employment_score",
        "Employment Decile (where 1 is most deprived 10% of LSOAs)": "employment_decile",
        "Health Deprivation and Disability Score": "health_deprivation_score",
        "Health Deprivation and Disability Decile (where 1 is most deprived)": "health_deprivation_decile",
        "Barriers to Housing and Services Score": "barriers_housing_services_score",
        "Barriers to Housing and Services Decile (where 1 is most deprived)": "barriers_housing_services_decile",
        "Living Environment Score": "living_environment_score",
        "Living Environment Decile (where 1 is most deprived)": "living_environment_decile",
    }

    collapsed: list[dict[str, object]] = []
    for code, group in sorted(grouped.items()):
        weights = [to_float(row.get(weight_col)) or 1.0 for row in group]
        total_weight = sum(weights) or float(len(group))
        out: dict[str, object] = {
            "LSOA21CD": code,
            "LSOA21NM": group[0].get("LSOA21NM", ""),
            "LAD22CD_from_imd_lookup": group[0].get("LAD22CD", ""),
            "LAD22NM_from_imd_lookup": group[0].get("LAD22NM", ""),
            "LAD2019CD_from_imd": group[0].get("Local Authority District code (2019)", ""),
            "LAD2019NM_from_imd": group[0].get("Local Authority District name (2019)", ""),
            "imd_source_rows_n": len(group),
            "imd_chgind_values": ";".join(sorted({row.get("CHGIND", "") for row in group if row.get("CHGIND", "")})),
            "imd_lsoa11_sources": ";".join(sorted({row.get("LSOA code (2011)", "") for row in group})),
        }
        for source_col, out_col in wanted.items():
            values = [to_float(row.get(source_col)) for row in group]
            valid = [(v, w) for v, w in zip(values, weights) if v is not None]
            if valid:
                out[f"{out_col}_pop_weighted"] = sum(v * w for v, w in valid) / total_weight
                out[f"{out_col}_min"] = min(v for v, _ in valid)
                out[f"{out_col}_max"] = max(v for v, _ in valid)
                if len(valid) == 1 or math.isclose(out[f"{out_col}_min"], out[f"{out_col}_max"]):
                    out[out_col] = valid[0][0]
                else:
                    out[out_col] = out[f"{out_col}_pop_weighted"]
        collapsed.append(out)

    return collapsed, {row["LSOA21CD"]: row for row in collapsed}


def build_census_context() -> tuple[list[dict[str, object]], dict[str, dict[str, object]]]:
    context_by_lsoa: dict[str, dict[str, object]] = {}
    for path in sorted(CENSUS_DIR.glob("*_lcr_lsoa2021.csv")):
        table_code = path.name.split("_", 1)[0].lower()
        rows = read_csv(path)
        for row in rows:
            code = row.get("geography code")
            if not code:
                continue
            context = context_by_lsoa.setdefault(
                code,
                {
                    "LSOA21CD": code,
                    "LSOA21NM_census": row.get("geography", ""),
                },
            )
            for key, value in row.items():
                if key in {"date", "geography", "geography code"}:
                    continue
                context[f"{table_code}_{slug(key)}"] = value

    for context in context_by_lsoa.values():
        total_pop = to_float(context.get("ts001_residence_type_total_measures_value"))
        households = to_float(context.get("ts011_household_deprivation_total_all_households_measures_value"))
        deprived_2 = to_float(context.get("ts011_household_deprivation_household_is_deprived_in_two_dimensions_measures_value"))
        deprived_3 = to_float(context.get("ts011_household_deprivation_household_is_deprived_in_three_dimensions_measures_value"))
        deprived_4 = to_float(context.get("ts011_household_deprivation_household_is_deprived_in_four_dimensions_measures_value"))
        car_total = to_float(context.get("ts045_number_of_cars_or_vans_total_all_households"))
        no_car = to_float(context.get("ts045_number_of_cars_or_vans_no_cars_or_vans_in_household"))
        travel_total = to_float(
            context.get(
                "ts061_method_of_travel_to_workplace_total_all_usual_residents_aged_16_years_and_over_in_employment_the_week_before_the_census"
            )
        )
        context["census_usual_residents"] = total_pop
        context["census_population_density_ppsqkm"] = to_float(
            context.get("ts006_population_density_persons_per_square_kilometre_measures_value")
        )
        context["census_pct_households_deprived_2plus_dims"] = safe_div(
            (deprived_2 or 0) + (deprived_3 or 0) + (deprived_4 or 0), households
        )
        context["census_pct_households_no_car"] = safe_div(no_car, car_total)
        context["census_pct_commute_bicycle"] = safe_div(
            to_float(context.get("ts061_method_of_travel_to_workplace_bicycle")), travel_total
        )
        context["census_pct_commute_on_foot"] = safe_div(
            to_float(context.get("ts061_method_of_travel_to_workplace_on_foot")), travel_total
        )
        context["census_pct_commute_car_driver"] = safe_div(
            to_float(context.get("ts061_method_of_travel_to_workplace_driving_a_car_or_van")), travel_total
        )
        context["census_pct_work_from_home"] = safe_div(
            to_float(context.get("ts061_method_of_travel_to_workplace_work_mainly_at_or_from_home")), travel_total
        )

    rows = [context_by_lsoa[code] for code in sorted(context_by_lsoa)]
    return rows, context_by_lsoa


def merge_lsoa_context() -> tuple[list[dict[str, object]], dict[str, dict[str, object]], dict[str, object]]:
    lsoa_lookup = build_lsoa_lookup()
    imd_rows, imd_by_lsoa = build_imd_lsoa21()
    census_rows, census_by_lsoa = build_census_context()

    all_codes = sorted(set(lsoa_lookup) | set(imd_by_lsoa) | set(census_by_lsoa))
    rows: list[dict[str, object]] = []
    for code in all_codes:
        row: dict[str, object] = {
            "analysis_spatial_unit": "LSOA 2021",
            "analysis_lsoa21cd": code,
        }
        lookup = lsoa_lookup.get(code, {})
        row["analysis_lsoa21nm"] = lookup.get("LSOA21NM") or lookup.get("lsoa21nm") or ""
        row["LAD22CD"] = ""
        row["LAD22NM"] = ""
        row["context_lsoa_lookup_joined"] = code in lsoa_lookup
        row["context_census_joined"] = code in census_by_lsoa
        row["context_imd_joined"] = code in imd_by_lsoa
        row.update(census_by_lsoa.get(code, {}))
        row.update(imd_by_lsoa.get(code, {}))
        row["LAD22CD"] = row.get("LAD22CD_from_imd_lookup", "")
        row["LAD22NM"] = row.get("LAD22NM_from_imd_lookup", "")
        rows.append(row)

    qa = {
        "lsoa_lookup_rows": len(lsoa_lookup),
        "census_lsoa_rows": len(census_by_lsoa),
        "imd_lsoa21_rows_after_collapse": len(imd_by_lsoa),
        "combined_lsoa_context_rows": len(rows),
        "census_missing_from_lsoa_lookup": len(set(census_by_lsoa) - set(lsoa_lookup)),
        "imd_missing_from_lsoa_lookup": len(set(imd_by_lsoa) - set(lsoa_lookup)),
        "lsoa_lookup_missing_census": len(set(lsoa_lookup) - set(census_by_lsoa)),
        "lsoa_lookup_missing_imd": len(set(lsoa_lookup) - set(imd_by_lsoa)),
        "imd_rows_with_multiple_lsoa11_sources": sum(1 for row in imd_rows if int(row["imd_source_rows_n"]) > 1),
    }
    return rows, {row["analysis_lsoa21cd"]: row for row in rows}, qa


def build_sensor_context(sensor_spatial: list[dict[str, object]], lsoa_context: dict[str, dict[str, object]]) -> list[dict[str, object]]:
    rows = []
    for row in sensor_spatial:
        code = row.get("analysis_lsoa21cd")
        context = lsoa_context.get(str(code), {})
        out = dict(row)
        out.update({k: v for k, v in context.items() if k not in out or k.startswith("census_") or k.startswith("imd_")})
        out["context_join_success"] = bool(context)
        out["census_join_success"] = bool(context.get("context_census_joined"))
        out["imd_join_success"] = bool(context.get("context_imd_joined"))
        rows.append(out)
    return rows


def build_daily_context(sensor_context_rows: list[dict[str, object]]) -> tuple[list[dict[str, object]], dict[str, object]]:
    if not DAILY_SENSOR_MAIN.exists():
        return [], {"daily_source_exists": False}

    context_by_countline = {str(row["countline_id"]): row for row in sensor_context_rows}
    selected_context_cols = [
        "analysis_spatial_unit",
        "analysis_lsoa21cd",
        "analysis_lsoa21nm",
        "LAD22NM",
        "lsoa21_rural_urban_class",
        "imd_decile",
        "imd_score",
        "income_decile",
        "income_score",
        "employment_decile",
        "health_deprivation_decile",
        "living_environment_decile",
        "census_usual_residents",
        "census_population_density_ppsqkm",
        "census_pct_households_deprived_2plus_dims",
        "census_pct_households_no_car",
        "census_pct_commute_bicycle",
        "census_pct_commute_on_foot",
        "census_pct_commute_car_driver",
        "census_pct_work_from_home",
    ]

    daily_rows = read_csv(DAILY_SENSOR_MAIN)
    joined_rows: list[dict[str, object]] = []
    unmatched = 0
    for row in daily_rows:
        context = context_by_countline.get(str(row.get("countline_id")))
        out = dict(row)
        if context:
            for col in selected_context_cols:
                out[col] = context.get(col, "")
            out["daily_context_join_success"] = True
        else:
            unmatched += 1
            out["daily_context_join_success"] = False
        joined_rows.append(out)

    qa = {
        "daily_source_exists": True,
        "daily_rows": len(daily_rows),
        "daily_joined_rows": len(joined_rows),
        "daily_unmatched_rows": unmatched,
        "daily_unique_countlines": len({row.get("countline_id") for row in daily_rows}),
    }
    return joined_rows, qa


def main() -> None:
    SENSOR_OUT.mkdir(parents=True, exist_ok=True)
    QA_OUT.mkdir(parents=True, exist_ok=True)

    sensor_locations = build_countline_locations()
    sensor_spatial = spatial_join_lsoa(sensor_locations)
    imd_rows, _ = build_imd_lsoa21()
    lsoa_context_rows, lsoa_context_by_code, lsoa_context_qa = merge_lsoa_context()
    sensor_context_rows = build_sensor_context(sensor_spatial, lsoa_context_by_code)
    daily_context_rows, daily_context_qa = build_daily_context(sensor_context_rows)

    write_csv(SENSOR_OUT / "vivacity_countline_locations.csv", sensor_locations)
    write_csv(SENSOR_OUT / "vivacity_countline_lsoa2021_spatial_join.csv", sensor_spatial)
    write_csv(PROCESSED / "imd" / "imd_2019_lcr_lsoa2021_collapsed.csv", imd_rows)
    write_csv(PROCESSED / "lsoa" / "lsoa2021_lcr_context_wide.csv", lsoa_context_rows)
    write_csv(SENSOR_OUT / "vivacity_countline_lsoa2021_context_joined.csv", sensor_context_rows)
    if daily_context_rows:
        write_csv(SENSOR_OUT / "vivacity_daily_lsoa2021_context_analysis.csv", daily_context_rows)

    total_sensors = len(sensor_spatial)
    spatial_matched = sum(1 for row in sensor_spatial if row.get("spatial_join_success"))
    context_matched = sum(1 for row in sensor_context_rows if row.get("context_join_success"))
    census_matched = sum(1 for row in sensor_context_rows if row.get("census_join_success"))
    imd_matched = sum(1 for row in sensor_context_rows if row.get("imd_join_success"))
    unique_lsoas = len({row.get("analysis_lsoa21cd") for row in sensor_spatial if row.get("analysis_lsoa21cd")})

    qa_rows = [
        {
            "check": "synchronised_spatial_unit",
            "result": "LSOA 2021",
            "passed": True,
            "notes": "Vivacity countline GPS midpoints spatially joined to LSOA 2021; Census 2021 is native LSOA 2021; IMD 2019 bridged from LSOA 2011 via exact-fit lookup.",
        },
        {"check": "target_countlines", "result": total_sensors, "passed": total_sensors == 21, "notes": "Expected 21 dissertation countlines."},
        {"check": "spatial_join_matched_countlines", "result": spatial_matched, "passed": spatial_matched == total_sensors, "notes": "Countlines matched to an LSOA 2021 polygon."},
        {"check": "unique_lsoa21_areas_for_countlines", "result": unique_lsoas, "passed": unique_lsoas > 0, "notes": "Several countlines share the same road/scheme LSOA."},
        {"check": "context_join_matched_countlines", "result": context_matched, "passed": context_matched == total_sensors, "notes": "Countline LSOA code found in combined context table."},
        {"check": "census_join_matched_countlines", "result": census_matched, "passed": census_matched == total_sensors, "notes": "Countline LSOA code found in Census 2021 tables."},
        {"check": "imd_join_matched_countlines", "result": imd_matched, "passed": imd_matched == total_sensors, "notes": "Countline LSOA code found in IMD 2019 to LSOA 2021 bridge."},
    ]
    qa_rows.extend(
        {
            "check": key,
            "result": value,
            "passed": value == 0 if key in {"lsoa_lookup_missing_census", "lsoa_lookup_missing_imd"} else True,
            "notes": "Extra bridged IMD rows outside the LCR boundary are acceptable if all boundary LSOAs and sensor LSOAs match."
            if key == "imd_missing_from_lsoa_lookup"
            else "",
        }
        for key, value in lsoa_context_qa.items()
    )
    qa_rows.extend(
        {
            "check": key,
            "result": value,
            "passed": value == 0 if key == "daily_unmatched_rows" else True,
            "notes": "Daily Vivacity rows joined to the countline-level LSOA 2021 context table.",
        }
        for key, value in daily_context_qa.items()
    )
    write_csv(QA_OUT / "sensor_lsoa_context_join_qa.csv", qa_rows, ["check", "result", "passed", "notes"])

    unmatched = [
        row
        for row in sensor_context_rows
        if not row.get("spatial_join_success") or not row.get("context_join_success") or not row.get("census_join_success") or not row.get("imd_join_success")
    ]
    write_csv(QA_OUT / "sensor_lsoa_context_unmatched_rows.csv", unmatched)

    summary = {
        "spatial_unit_chosen": "LSOA 2021",
        "target_countlines": total_sensors,
        "spatial_matched": spatial_matched,
        "context_matched": context_matched,
        "census_matched": census_matched,
        "imd_matched": imd_matched,
        "unique_lsoa21_areas": unique_lsoas,
        **daily_context_qa,
        "outputs_folder": str(SENSOR_OUT),
    }
    (QA_OUT / "sensor_lsoa_context_join_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()
